In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))


In [2]:
import src.preprocessing as prep

print(dir(prep))

['AmountTimeScaler', 'BaseEstimator', 'StandardScaler', 'TransformerMixin', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'np', 'pd', 'split_x_y']


In [3]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from src.preprocessing import AmountTimeScaler, split_x_y

DATA_PROCESSED = Path("../data/processed")
train = pd.read_csv(DATA_PROCESSED / "batch1_train.csv")
test  = pd.read_csv(DATA_PROCESSED / "batch2_test.csv")
stream1 = pd.read_csv(DATA_PROCESSED / "batch3_stream.csv")
stream2 = None
if (DATA_PROCESSED / "batch4_stream.csv").exists():
    stream2 = pd.read_csv(DATA_PROCESSED / "batch4_stream.csv")


#Fit Baseline

In [4]:
x_tr, y_tr = split_x_y(train)
x_te, y_te = split_x_y(test)

print(x_tr.select_dtypes(include=['object', 'string']).columns.tolist())


['transaction_id', 'timestamp', 'sender_account', 'receiver_account', 'transaction_type', 'merchant_category', 'location', 'device_used', 'fraud_type', 'payment_channel', 'ip_address', 'device_hash']


In [5]:
for col in x_tr.select_dtypes(include=["object","string"]).columns:
    print(col, x_tr[col].nunique())

transaction_id 125000
timestamp 125000
sender_account 116717
receiver_account 116639
transaction_type 4
merchant_category 8
location 8
device_used 4
fraud_type 1
payment_channel 4
ip_address 124998
device_hash 124137


In [6]:
high_cardinality_cols = [
    "transaction_id",
    "sender_account",
    "receiver_account",
    "ip_address",
    "device_hash"
]

x_tr = x_tr.drop(columns=high_cardinality_cols, errors="ignore")
x_te = x_te.drop(columns=high_cardinality_cols, errors="ignore")

In [7]:
print(x_tr.dtypes)

timestamp                          str
amount                         float64
transaction_type                   str
merchant_category                  str
location                           str
device_used                        str
fraud_type                         str
time_since_last_transaction    float64
spending_deviation_score       float64
velocity_score                   int64
geo_anomaly_score              float64
payment_channel                    str
dtype: object


In [8]:
categorical_cols = [
    "transaction_type",
    "merchant_category",
    "location",
    "device_used",
    "payment_channel"
]

x_tr = pd.get_dummies(
    x_tr,
    columns=categorical_cols,
    drop_first=True
)

x_te = pd.get_dummies(
    x_te,
    columns=categorical_cols,
    drop_first=True
)

x_tr, x_te = x_tr.align(
    x_te,
    join="left",
    axis=1,
    fill_value=0
)

for col in categorical_cols:
    print(col, col in x_tr.columns)

transaction_type False
merchant_category False
location False
device_used False
payment_channel False


In [9]:
for col in train.columns:
    print(col, train[col].nunique())

transaction_id 125000
timestamp 125000
sender_account 116717
receiver_account 116639
amount 60862
transaction_type 4
merchant_category 8
location 8
device_used 4
is_fraud 2
fraud_type 1
time_since_last_transaction 102603
spending_deviation_score 744
velocity_score 20
geo_anomaly_score 101
payment_channel 4
ip_address 124998
device_hash 124137


In [10]:
print(x_tr.select_dtypes(include=['object', 'string']).columns.tolist())

['timestamp', 'fraud_type']


In [11]:
drop_cols = [
    "transaction_id",
    "timestamp",        # after extracting hour/day features
    "ip_address",
    "device_hash",
    "sender_account",
    "receiver_account",
    "fraud_type"
]

In [12]:
x_tr["timestamp"] = pd.to_datetime(x_tr["timestamp"], format="ISO8601")
x_te["timestamp"] = pd.to_datetime(x_te["timestamp"], format="ISO8601")

for df in [x_tr, x_te]:
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [13]:
x_tr = x_tr.drop(columns=drop_cols, errors="ignore")
x_te = x_te.drop(columns=drop_cols, errors="ignore")

In [14]:
print(x_tr.shape)
print(x_tr.columns[:20])

(125000, 31)
Index(['amount', 'time_since_last_transaction', 'spending_deviation_score',
       'velocity_score', 'geo_anomaly_score', 'transaction_type_payment',
       'transaction_type_transfer', 'transaction_type_withdrawal',
       'merchant_category_grocery', 'merchant_category_online',
       'merchant_category_other', 'merchant_category_restaurant',
       'merchant_category_retail', 'merchant_category_travel',
       'merchant_category_utilities', 'location_Dubai', 'location_London',
       'location_New York', 'location_Singapore', 'location_Sydney'],
      dtype='str')


In [15]:
print(x_tr.columns.tolist())

['amount', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'transaction_type_payment', 'transaction_type_transfer', 'transaction_type_withdrawal', 'merchant_category_grocery', 'merchant_category_online', 'merchant_category_other', 'merchant_category_restaurant', 'merchant_category_retail', 'merchant_category_travel', 'merchant_category_utilities', 'location_Dubai', 'location_London', 'location_New York', 'location_Singapore', 'location_Sydney', 'location_Tokyo', 'location_Toronto', 'device_used_mobile', 'device_used_pos', 'device_used_web', 'payment_channel_UPI', 'payment_channel_card', 'payment_channel_wire_transfer', 'hour', 'day_of_week', 'is_weekend']


In [16]:
print(x_tr.dtypes.value_counts())

bool       23
float64     4
int64       2
int32       2
Name: count, dtype: int64


In [17]:
print(x_tr.select_dtypes(include=['object', 'string']).columns.tolist())

[]


In [18]:
categorical_cols = [
    "merchant_category",
    "location",
    "device_used",
    "transaction_type",
    "payment_channel"
]

In [19]:
print(x_tr.shape)

(125000, 31)


In [20]:
print(x_tr.shape)
print(x_tr.select_dtypes(include=["object", "string"]).columns.tolist())

(125000, 31)
[]


In [21]:
import inspect
print(inspect.getsource(split_x_y))

def split_x_y(df, label_col="is_fraud"):
    y = df[label_col].astype(int).values
    x = df.drop(columns=[label_col])
    return x, y



In [22]:
#SPLIT X AND Y

x_tr, y_tr = split_x_y(train)
x_te, y_te = split_x_y(test)

In [23]:
#DROP HIGH-CARDINALITY COLUMNS

high_cardinality_cols = [
    "transaction_id",
    "sender_account",
    "receiver_account",
    "ip_address",
    "device_hash"
]

x_tr = x_tr.drop(columns=high_cardinality_cols, errors="ignore")
x_te = x_te.drop(columns=high_cardinality_cols, errors="ignore")

In [24]:
#CONVERT TIMESTAMP

x_tr["timestamp"] = pd.to_datetime(x_tr["timestamp"], format="ISO8601")
x_te["timestamp"] = pd.to_datetime(x_te["timestamp"], format="ISO8601")

In [25]:
#CREATE TIME FEATURES
for df in [x_tr, x_te]:
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [26]:
#DROP TIMESTAMP AND FRAUD_TYPE

drop_cols = [
    "timestamp",
    "fraud_type"
]

x_tr = x_tr.drop(columns=drop_cols, errors="ignore")
x_te = x_te.drop(columns=drop_cols, errors="ignore")

In [27]:
#ONE-HOT ENCODE CATEGORICAL COLUMNS

categorical_cols = [
    "transaction_type",
    "merchant_category",
    "location",
    "device_used",
    "payment_channel"
]

x_tr = pd.get_dummies(
    x_tr,
    columns=categorical_cols,
    drop_first=True
)

x_te = pd.get_dummies(
    x_te,
    columns=categorical_cols,
    drop_first=True
)

In [28]:
#ALIGN TRAIN AND TEST

x_tr, x_te = x_tr.align(
    x_te,
    join="left",
    axis=1,
    fill_value=0
)

In [29]:
#VERIFY PREPROCESSING

print(x_tr.shape)
print(x_tr.select_dtypes(include=["object", "string"]).columns.tolist())

(125000, 31)
[]


In [30]:
x_tr.isna().sum().sort_values(ascending=False)

time_since_last_transaction      22397
amount                               0
spending_deviation_score             0
velocity_score                       0
geo_anomaly_score                    0
hour                                 0
day_of_week                          0
is_weekend                           0
transaction_type_payment             0
transaction_type_transfer            0
transaction_type_withdrawal          0
merchant_category_grocery            0
merchant_category_online             0
merchant_category_other              0
merchant_category_restaurant         0
merchant_category_retail             0
merchant_category_travel             0
merchant_category_utilities          0
location_Dubai                       0
location_London                      0
location_New York                    0
location_Singapore                   0
location_Sydney                      0
location_Tokyo                       0
location_Toronto                     0
device_used_mobile       

In [31]:
print(x_tr.isna().sum().sum())

22397


In [32]:
na_cols = x_tr.columns[x_tr.isna().any()]

for col in na_cols:
    print(col, x_tr[col].isna().sum())

time_since_last_transaction 22397


In [33]:
x_tr = x_tr.fillna(x_tr.median(numeric_only=True))
x_te = x_te.fillna(x_tr.median(numeric_only=True))

In [34]:
#scale numeric feature

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    ))
])
print(x_tr.shape)
print(x_tr.select_dtypes(include=["object", "string"]).columns.tolist())
pipe.fit(x_tr, y_tr)

(125000, 31)
[]


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['amount','time_since_last_transaction','spending_deviation_score',..., 'payment_channel_UPI','payment_channel_card', 'payment_channel_wire_transfer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [47]:
#to check model is usable or not

proba_test = pipe.predict_proba(x_te)[:,1]
print(proba_test[:10])

[0.48157489 0.4630238  0.48313096 0.520853   0.48598075 0.51527346
 0.49342972 0.49558199 0.52112825 0.50361419]


In [48]:
#How many iterations Logistic Regression actually used before stopping

print(pipe.named_steps["clf"].n_iter_)

[10]


In [45]:
print(pipe.score(x_tr, y_tr))

0.51248


#Default threshold = 0.5 metrics for baseline model

In [49]:
def metrics_at_threshold(y_true, proba, thr=0.5):
    y_pred = (proba >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    ap = average_precision_score(y_true, proba)  # PR-AUC
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"threshold":thr, "precision":p, "recall":r, "f1":f1, "pr_auc":ap,
            "TP":tp, "FP":fp, "TN":tn, "FN":fn}

baseline = metrics_at_threshold(y_te, proba_test, 0.5)
pd.DataFrame([baseline])


,threshold,precision,recall,f1,pr_auc,TP,FP,TN,FN
0,0.5,0.034872,0.455182,0.064781,0.035893,2016,55795,64776,2413


#Target-recall threshold + “manual review” workload

#Finding threshold for a target recall

In [51]:
def threshold_for_target_recall(y_true, proba, target_recall=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # precision_recall_curve returns thresholds len-1 compared to recalls
    # Find first threshold where recall >= target_recall scanning from high recall
    idx = np.where(recalls >= target_recall)[0]
    if len(idx)==0:
        # can't reach target recall; fall back to min threshold
        chosen_thr = thresholds.min() if len(thresholds)>0 else 0.0
    else:
        i = idx[-1] - 1  # align to thresholds indexing
        i = max(0, min(i, len(thresholds)-1))
        chosen_thr = thresholds[i]
    return float(chosen_thr)

target_recall = 0.80
thr = threshold_for_target_recall(y_te, proba_test, target_recall)
thr


0.47946946399271295

#Report recall & workload at that threshold

In [52]:
def review_load(y_true, proba, thr):
    y_pred = (proba >= thr).astype(int)
    flagged = y_pred.sum()
    total = len(y_true)
    return flagged, flagged/total

m = metrics_at_threshold(y_te, proba_test, thr)
flagged, frac = review_load(y_te, proba_test, thr)

print(f"Chosen threshold for ~{target_recall:.0%} recall: {thr:.4f}")
print(f"TEST metrics @thr: precision={m['precision']:.3f}, recall={m['recall']:.3f}, F1={m['f1']:.3f}, PR-AUC={m['pr_auc']:.3f}")
print(f"Manual review load: {flagged} / {len(y_te)} ({frac:.2%}) flagged")
pd.DataFrame([m])


Chosen threshold for ~80% recall: 0.4795
TEST metrics @thr: precision=0.035, recall=0.800, F1=0.067, PR-AUC=0.036
Manual review load: 100857 / 125000 (80.69%) flagged


,threshold,precision,recall,f1,pr_auc,TP,FP,TN,FN
0,0.479469,0.035139,0.800181,0.067321,0.035893,3544,97313,23258,885


#Basic drift check

In [57]:
print(x_tr.shape)
print(x_tr.columns.tolist()[:15])

(125000, 31)
['amount', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'hour', 'day_of_week', 'is_weekend', 'transaction_type_payment', 'transaction_type_transfer', 'transaction_type_withdrawal', 'merchant_category_grocery', 'merchant_category_online', 'merchant_category_other', 'merchant_category_restaurant']


In [58]:
train_columns = x_tr.columns

In [59]:
def preprocess_features(x):

    high_cardinality_cols = [
        "transaction_id",
        "sender_account",
        "receiver_account",
        "ip_address",
        "device_hash"
    ]

    x = x.drop(columns=high_cardinality_cols, errors="ignore")

    x["timestamp"] = pd.to_datetime(x["timestamp"], format="ISO8601")

    x["hour"] = x["timestamp"].dt.hour
    x["day_of_week"] = x["timestamp"].dt.dayofweek
    x["is_weekend"] = (x["day_of_week"] >= 5).astype(int)

    x = x.drop(
        columns=["timestamp", "fraud_type"],
        errors="ignore"
    )

    categorical_cols = [
        "transaction_type",
        "merchant_category",
        "location",
        "device_used",
        "payment_channel"
    ]

    x = pd.get_dummies(
        x,
        columns=categorical_cols,
        drop_first=True
    )

    return x

In [66]:
def evaluate_split(df, pipe, thr):

    x, y = split_x_y(df)

    x = preprocess_features(x)

    x = x.reindex(
        columns=train_columns,
        fill_value=0
    )

    x = x.fillna(0)

    proba = pipe.predict_proba(x)[:,1]

    m = metrics_at_threshold(y, proba, thr=thr)

    flagged, frac = review_load(y, proba, thr)

    m["flagged"] = flagged
    m["flagged_pct"] = frac

    return pd.Series(m)

In [67]:
rows = []
rows.append(evaluate_split(test, pipe, thr).rename("TEST"))
rows.append(evaluate_split(stream1, pipe, thr).rename("STREAM1"))

if stream2 is not None:
    rows.append(evaluate_split(stream2, pipe, thr).rename("STREAM2"))

pd.concat(rows, axis=1)

,TEST,STREAM1,STREAM2
threshold,0.479469,0.479469,0.479469
precision,0.035571,0.036145,0.036028
recall,0.800181,0.800134,0.810472
f1,0.068115,0.069165,0.068989
pr_auc,0.037020,0.037125,0.036553
TP,3544.000000,3591.000000,3622.000000
FP,96087.000000,95759.000000,96912.000000
TN,24484.000000,24753.000000,23619.000000
FN,885.000000,897.000000,847.000000
flagged,99631.000000,99350.000000,100534.000000


In [64]:
try:
    evaluate_split(test, pipe, thr)
except Exception as e:
    print(type(e))
    print(e)

<class 'ValueError'>
Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values


In [65]:
x, y = split_x_y(test)

x = preprocess_features(x)

x = x.reindex(
    columns=train_columns,
    fill_value=0
)

print(x.isna().sum()[x.isna().sum() > 0])

time_since_last_transaction    22431
dtype: int64
